### Import ans setUp

In [11]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ['GROQ_API_KEY']:
    print("Api key is set")

Api key is set


In [12]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

model = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.0
)

### **RAG Implementation with PDF**

#### **Step 1 : Extracting text from Pdf**

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "./Docs/Lora.pdf"

loader = PyPDFLoader(pdf_path)

docs = loader.load()
docs


#### **Step 2 : Splitting the documents into Chuncks**

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, # chunk size
    chunk_overlap = 100 # take 100 character from previous chunk so data flow cant  be break
)

chunks = splitter.split_documents(docs)
chunks


In [ ]:
chunks[0].metadata

#### **Step 3 : Creating the embeddings from chunks**

In [7]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

C:\Users\Star\AppData\Local\Temp\ipykernel_19488\384917042.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### **Step 4: Create and Store a Embeddings in Vector Store**

In [8]:
from langchain_community.vectorstores import Chroma

vectoreStore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)

### **Similarity Search**

In [ ]:
vectoreStore.similarity_search("what is lora?", k=3)

### **Talk To LLM**

In [14]:
context = vectoreStore.similarity_search("what is Lora?", k=3)

response = model.invoke(f"what is Lora?, you can aswer using following context :{context}")
print(response.content)

Based on the provided context, LoRA stands for Low-Rank Adaptation. It is a method for adapting pre-trained models to new tasks without requiring the accumulated gradient update to weight matrices to have full-rank during adaptation.

In simpler terms, LoRA is a technique used in deep learning to fine-tune pre-trained models for specific tasks without having to retrain the entire model from scratch. It achieves this by adding trainable pairs of rank decomposition matrices in parallel to the existing weight matrices of the model.

The key benefits of LoRA include:

1. **Greater efficiency**: LoRA requires fewer trainable parameters compared to other adapter-based methods, making it more computationally efficient.
2. **No additional inference latency**: LoRA allows for explicit computation and storage of the modified weight matrices, which can be used for inference without any additional latency.
3. **Improved performance**: LoRA has been shown to perform better than or at least on-par w